In [56]:
import os
import cv2
from ultralytics import YOLO
from pathlib import Path
import subprocess
import ffmpeg
import cv2
import numpy as np
import random
import matplotlib.pyplot as plt
import shutil

In [58]:
images_path = "dataset/train/images"
labels_path = "dataset/train/labels"
val_images_path = "dataset/val/images"
val_labels_path = "dataset/train/labels"

os.makedirs(val_images_path, exist_ok=True)
os.makedirs(val_labels_path, exist_ok=True)

image_files = [f for f in os.listdir(images_path) if f.endswith(".jpg")]

val_files = random.sample(image_files, k=int(len(image_files) * 0.15))

for file_name in val_files:
    shutil.move(
        os.path.join(images_path, file_name), os.path.join(val_images_path, file_name)
    )

    label_file = file_name.replace(".jpg", ".txt")
    if os.path.exists(os.path.join(labels_path, label_file)):
        shutil.move(
            os.path.join(labels_path, label_file),
            os.path.join(val_labels_path, label_file),
        )

print(f"Перенесено {len(val_files)} изображений и меток в папку валидации.")

Перенесено 68 изображений и меток в папку валидации.


In [59]:
import yaml

dataset_path = Path("dataset")


data = {
    "path": "/Users/magewade/Desktop/ML/puppies_detection/dataset",
    "train": "train",
    "val": "val",
    "names": {"0": "dog"},
}

# Путь для сохранения файла
yaml_path = dataset_path / "data.yaml"

# Запись данных в YAML
with open(yaml_path, "w") as yaml_file:
    yaml.dump(data, yaml_file, default_flow_style=False)

print(f"Файл data.yaml успешно создан в {yaml_path}")

Файл data.yaml успешно создан в dataset/data.yaml


In [ ]:
model = YOLO("yolov8n.pt")

model.train(
    data="dataset/data.yaml",
    project="runs",
    epochs=50,
    imgsz=416,
    device="mps",
    augment=False,
    workers=1,
    batch=4,
)

In [2]:
model = YOLO("/Users/magewade/Desktop/ML/puppies_detection/best.pt")

In [40]:
conf = 0.02
iou = 0.7

In [39]:
import subprocess
import numpy as np
from ultralytics import YOLO
import yt_dlp
import cv2


# 🎯 Получаем прямую ссылку на видеопоток + размер кадра
def get_stream_info(youtube_url):
    ydl_opts = {
        "quiet": True,
        "format": "best[ext=mp4]/best",
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(youtube_url, download=False)
        width = info.get("width", 480)
        height = info.get("height", 290)
        return info["url"], width, height


# 📺 Ссылка на YouTube
youtube_url = "https://www.youtube.com/watch?v=bYlEgU2tU5w"
stream_url, frame_width, frame_height = get_stream_info(youtube_url)

print(f"📺 Получен поток: {stream_url}")
print(f"📐 Размер кадра: {frame_width}x{frame_height}")

# 🧵 Настройка ffmpeg
ffmpeg_cmd = [
    "ffmpeg",
    "-i",
    stream_url,
    "-vf",
    f"scale={frame_width}:{frame_height}",  # убедимся, что размер фиксированный
    "-f",
    "image2pipe",
    "-pix_fmt",
    "bgr24",
    "-vcodec",
    "rawvideo",
    "-loglevel",
    "quiet",
    "-",
]
pipe = subprocess.Popen(ffmpeg_cmd, stdout=subprocess.PIPE)

frame_size = frame_width * frame_height * 3
frame_count = 0
skip_every = 1  # обрабатывать каждый 5-й кадр
results = None  # последние результаты YOLO

try:
    while True:
        raw_frame = pipe.stdout.read(frame_size)
        if not raw_frame:
            print("🚫 Поток завершился или прервался")
            break

        frame = np.frombuffer(raw_frame, dtype=np.uint8)
        if frame.size != frame_size:
            print("⚠️ Размер кадра не совпадает, пропуск...")
            continue

        frame = frame.reshape((frame_height, frame_width, 3))

        frame_count += 1
        if frame_count % skip_every == 0:
            results = model.track(
                source=frame,
                persist=True,
                conf=conf,
                iou=iou,
                tracker="puppy_tracker.yaml",
                verbose=False,
            )

        if results:
            annotated = results[0].plot()
        else:
            annotated = frame

        cv2.imshow("YOLO Stream", annotated)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

except KeyboardInterrupt:
    print("⛔ Остановлено пользователем")

finally:
    pipe.terminate()
    cv2.destroyAllWindows()

📺 Получен поток: https://manifest.googlevideo.com/api/manifest/hls_playlist/expire/1744927048/ei/6CQBaNPhBY7M0u8PptGNsQc/ip/158.181.215.235/id/bYlEgU2tU5w.1/itag/96/source/yt_live_broadcast/requiressl/yes/ratebypass/yes/live/1/sgoap/gir%3Dyes%3Bitag%3D140/sgovp/gir%3Dyes%3Bitag%3D137/rqh/1/hls_chunk_host/rr3---sn-hxb5apox-4g0s.googlevideo.com/xpc/EgVo2aDSNQ%3D%3D/playlist_duration/30/manifest_duration/30/bui/AccgBcP5A8Wn29ucWNoJtgAPzR55ySrVXBkF7mrnOdSANuJ2huN8J2Mb80BQA8iNkodKN1LHOVVDjot5/spc/_S3wKvajoAfVsA4FF8x-r2B-bkBEROmpf6GtDybBdJpE5yJBxxyJFR9p872p1UtdS-VxZyU/vprv/1/playlist_type/DVR/initcwndbps/1682500/met/1744905449,/mh/uK/mm/44/mn/sn-hxb5apox-4g0s/ms/lva/mv/m/mvi/3/pcm2cms/yes/pl/20/rms/lva,lva/dover/11/pacing/0/keepalive/yes/fexp/51355912/mt/1744904918/sparams/expire,ei,ip,id,itag,source,requiressl,ratebypass,live,sgoap,sgovp,rqh,xpc,playlist_duration,manifest_duration,bui,spc,vprv,playlist_type/sig/AJfQdSswRQIgVa-qt-x75OMMYeqRRrJjI3wyc-9ot0VlDlkjQNfRwHACIQDvlNyFF6YynMfdbt3U6OkV

In [ ]:
results = model.track(
    source="/Users/magewade/Desktop/ML/puppies_detection/video/puppies_inference_2.mp4",
    conf=0.4,
    iou=0.4,
    tracker="puppy_tracker.yaml",
    show=True,
)


WARNING ⚠️ inference results will accumulate in RAM unless `stream=True` is passed, causing potential out-of-memory
errors for large sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs



KeyboardInterrupt: 

: 